In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Download Single "Acta de Escrutinio de Elección de Presidente" Pdf

In [ ]:
import asyncio
from playwright.async_api import async_playwright, Page

# Opening the browser and navigating to the page to select some options and click the "Next" button
async with async_playwright() as pw:

    # --- 1. Launch the browser -------------------------------------------
    # headless=False  → a real browser window opens so you can watch
    # slow_mo=800     → adds 800ms pause between actions
    # Change headless=True + remove slow_mo once everything works
    browser = await pw.chromium.launch(headless=False, slow_mo=800)
    page = await browser.new_page()
    # --- 2. Go to the website --------------------------------------------
    print("Navigating to site...")
    #await page.goto("https://books.toscrape.com/catalogue/page-1.html")
    await page.goto("https://divulgacione14presidente.registraduria.gov.co/departamento/72")

    # wait_for_load_state ensures the page is fully rendered before acting
    await page.wait_for_load_state("networkidle")

    print(f"  Title : {await page.title()}")
    print(f"  URL   : {page.url}")
    
    # wait_for_load_state ensures the page is fully rendered before acting
    await page.wait_for_load_state("networkidle")

    print(f"  Title : {await page.title()}")
    print(f"  URL   : {page.url}")
    
    # --- 4. Click the "Next" button --------------------------------------
    print("\nLooking for Municipio 006...")

Navigating to site...
  Title : Visor Ciudadano
  URL   : https://divulgacione14presidente.registraduria.gov.co/departamento/72
  Title : Visor Ciudadano
  URL   : https://divulgacione14presidente.registraduria.gov.co/departamento/72

Looking for Municipio 006...


## Reading Website and Select The First Option in Municipio

In [ ]:
"""
Opens the Registraduría Visor Ciudadano for a given department,
clicks the custom "Municipio" dropdown, and selects the first option.

The Municipio field is a custom Angular component — NOT a native <select>.
The click event is bound on the parent div.input-container, not the <input>
itself, so we click the container and dispatch events manually as fallback.
"""



# ── Config ─────────────────────────────────────────────────────────────────────
DEPARTMENT_ID = 72
HEADLESS      = False   # False = you can watch the browser
SLOW_MO       = 400
# ──────────────────────────────────────────────────────────────────────────────

URL = f"https://divulgacione14presidente.registraduria.gov.co/departamento/{DEPARTMENT_ID}"


async with async_playwright() as pw:
    browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)
    page = await browser.new_page()

    try:
        # ── 1. Navigate ────────────────────────────────────────────────────
        print(f"Opening: {URL}")
        await page.goto(URL)
        await page.wait_for_selector(
            "input[placeholder='seleccione el municipio']",
            timeout=15_000
        )
        print("Page loaded.")

        # ── 2. Locate the Municipio app-custom-select container ───────────
        # The Angular click handler lives on the div.input-container,
        # not on the <input> itself.
        municipio_container = page.locator("app-custom-select").filter(
            has=page.locator("input[placeholder='seleccione el municipio']")
        ).locator("div.input-container")

        # ── 3. Click the container div (where Angular binds the event) ─────
        print("Clicking Municipio container div...")
        await municipio_container.click()
        await page.wait_for_timeout(1000)

        # ── 4. Check if dropdown appeared; if not, try harder ──────────────
        if await page.locator(".dropdown-list li").count() == 0:
            print("Dropdown did not open with click — trying dispatchEvent...")
            await page.eval_on_selector(
                "input[placeholder='seleccione el municipio']",
                "el => { el.dispatchEvent(new MouseEvent('click', {bubbles: true})); }"
            )
            await page.wait_for_timeout(1000)

        if await page.locator(".dropdown-list li").count() == 0:
            print("Trying focus + input event...")
            await page.eval_on_selector(
                "input[placeholder='seleccione el municipio']",
                """el => {
                    el.dispatchEvent(new Event('focus', {bubbles: true}));
                    el.dispatchEvent(new MouseEvent('mousedown', {bubbles: true}));
                    el.dispatchEvent(new MouseEvent('mouseup', {bubbles: true}));
                    el.dispatchEvent(new MouseEvent('click', {bubbles: true}));
                }"""
            )
            await page.wait_for_timeout(1000)

        # ── 5. Wait for items to appear ────────────────────────────────────
        print("Waiting for dropdown items...")
        try:
            await page.wait_for_selector(
                ".dropdown-list li",
                state="visible",
                timeout=8_000
            )
            print("Dropdown is open!")
        except Exception:
            print("Trying search icon click as fallback...")
            icon = page.locator("app-custom-select").filter(
                has=page.locator("input[placeholder='seleccione el municipio']")
            ).locator("span.icon")
            await icon.click()
            await page.wait_for_selector(
                ".dropdown-list li",
                state="visible",
                timeout=8_000
            )
            print("Dropdown opened via icon click!")

        # ── 6. Scope to Municipio dropdown and read options ────────────────
        dropdown = page.locator("app-custom-select").filter(
            has=page.locator("input[placeholder='seleccione el municipio']")
        ).locator(".dropdown-list li")

        items = await dropdown.all()
        print(f"\nFound {len(items)} municipio(s):")
        for i, item in enumerate(items):
            text = (await item.inner_text()).strip()
            print(f"  [{i}] {text}")

        # ── 7. Select the first option ─────────────────────────────────────
        first_text = (await dropdown.first.inner_text()).strip()
        print(f"\nSelecting: '{first_text}'")
        await dropdown.first.click()

        # await page.wait_for_timeout(800)
        # selected = await page.locator(
        #     "input[placeholder='seleccione el municipio']"
        # ).input_value()
        # print(f"Selected value confirmed: '{selected}'")

        # print("\nDone!")

    finally:
        # Always close page before browser so in-flight network requests
        # are cancelled gracefully and don't produce TargetClosedError warnings
        await page.close()
        await browser.close()


Opening: https://divulgacione14presidente.registraduria.gov.co/departamento/72
Page loaded.
Clicking Municipio container div...
Waiting for dropdown items...
Dropdown is open!

Found 4 municipio(s):
  [0] 006 — CUMARIBO (100%)
  [1] 008 — LA PRIMAVERA (100%)
  [2] 001 — PUERTO CARREÑO (100%)
  [3] 002 — SANTA ROSALIA (100%)

Selecting: '006 — CUMARIBO (100%)'


## Selecting Municipio and Zona

In [6]:
DEPARTMENT_ID = 72
HEADLESS      = False
SLOW_MO       = 400

URL = f"https://divulgacione14presidente.registraduria.gov.co/departamento/{DEPARTMENT_ID}"


async def navigate(page: Page, url: str) -> None:
    """Go to the page and wait until the filter panel is ready."""
    print(f"Opening: {url}")
    await page.goto(url)
    await page.wait_for_selector(
        "input[placeholder='seleccione el municipio']",
        timeout=15_000
    )
    print("Page loaded.")


async def open_dropdown(page: Page, placeholder: str) -> None:
    """
    Click a custom Angular dropdown by its input placeholder and wait
    until the list items are visible. Tries progressively harder approaches
    if the initial click does not open the dropdown.
    """
    container = page.locator("app-custom-select").filter(
        has=page.locator(f"input[placeholder='{placeholder}']")
    ).locator("div.input-container")

    print(f"Clicking '{placeholder}' container...")
    await container.click()
    await page.wait_for_timeout(1000)

    if await page.locator(".dropdown-list li").count() == 0:
        print("  Dropdown did not open — trying dispatchEvent...")
        await page.eval_on_selector(
            f"input[placeholder='{placeholder}']",
            "el => el.dispatchEvent(new MouseEvent('click', {bubbles: true}))"
        )
        await page.wait_for_timeout(1000)

    if await page.locator(".dropdown-list li").count() == 0:
        print("  Trying full event sequence...")
        await page.eval_on_selector(
            f"input[placeholder='{placeholder}']",
            """el => {
                el.dispatchEvent(new Event('focus', {bubbles: true}));
                el.dispatchEvent(new MouseEvent('mousedown', {bubbles: true}));
                el.dispatchEvent(new MouseEvent('mouseup',  {bubbles: true}));
                el.dispatchEvent(new MouseEvent('click',    {bubbles: true}));
            }"""
        )
        await page.wait_for_timeout(1000)

    try:
        await page.wait_for_selector(".dropdown-list li", state="visible", timeout=8_000)
        print("  Dropdown is open!")
    except Exception:
        print("  Trying icon click as last fallback...")
        icon = page.locator("app-custom-select").filter(
            has=page.locator(f"input[placeholder='{placeholder}']")
        ).locator("span.icon")
        await icon.click()
        await page.wait_for_selector(".dropdown-list li", state="visible", timeout=8_000)
        print("  Dropdown opened via icon click!")


async def select_first_option(page: Page, placeholder: str) -> str:
    """
    Open a dropdown by placeholder, print all options, and click the first one.
    Returns the text of the selected option.
    """
    await open_dropdown(page, placeholder)

    dropdown = page.locator("app-custom-select").filter(
        has=page.locator(f"input[placeholder='{placeholder}']")
    ).locator(".dropdown-list li")

    items = await dropdown.all()
    print(f"  Found {len(items)} option(s):")
    for i, item in enumerate(items):
        print(f"    [{i}] {(await item.inner_text()).strip()}")

    first_text = (await dropdown.first.inner_text()).strip()
    print(f"  Selecting: '{first_text}'")
    await dropdown.first.click()
    await page.wait_for_timeout(800)

    return first_text


async def main() -> None:
    async with async_playwright() as pw:
        browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)
        page = await browser.new_page()

        try:
            await navigate(page, URL)

            print("\n── Municipio ──────────────────────────────────")
            municipio = await select_first_option(page, "seleccione el municipio")
            print(f"  ✓ Municipio selected: {municipio}")

            print("\n── Zona ───────────────────────────────────────")
            zona = await select_first_option(page, "seleccione la zona")
            print(f"  ✓ Zona selected: {zona}")

            print("\nDone!")

        finally:
            await page.close()
            await browser.close()
